# BB ブレイクアウト成功確率 LSTM 学習ノートブック

## 概要
ボリンジャーバンド +2σ をブレイクアウトした銘柄が、その後 **10営業日以内に +5% 上昇するか** を
PyTorch LSTM で予測します。

## 前提環境
- NVIDIA RTX 5080 (Blackwell, SM_120)
- CUDA 12.8 以上
- PyTorch 2.6+ (CUDA 12.8 版)

## インストール（初回のみ）
```bash
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128
pip install scikit-learn yfinance pandas numpy jupyter
```

## 学習後の成果物
- `models/lstm_bb.pt` — 学習済みモデルの重み
- `models/scaler_bb.pkl` — 特徴量スケーラー

これらを GitHub に push すると Streamlit Cloud の BB スキャナーで AI 確率が表示されます。

In [1]:
# ── セル 1: ライブラリ インポート ──────────────────────────────────────────
import sys
import os
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report

warnings.filterwarnings("ignore")

# kabu_dashboard/ ルートを sys.path に追加（modules/ をインポートできるようにする）
ROOT = Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from modules.lstm_model import LSTMClassifier, SEQ_LEN, N_FEATURES
from modules.indicators import calc_bollinger_bands, calc_volume_ma

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")

PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU             : NVIDIA GeForce RTX 5080


In [2]:
# ── セル 2: デバイス & ハイパーパラメータ ──────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用デバイス: {DEVICE}")

# --- 学習ハイパーパラメータ ---
BB_PERIOD     = 20       # ボリンジャーバンド期間
BB_THRESHOLD  = 0.97     # 上限到達判定（97%）
LOOKBACK      = 5        # 新規ブレイク判定（5日前は未達）
LABEL_HORIZON = 10       # ラベル: N 営業日以内に
LABEL_GAIN    = 0.05     # +5% 以上上昇 = 成功 (1)
SEQ_LEN_      = SEQ_LEN  # 特徴量シーケンス長（20日）

# --- 学習パラメータ ---
BATCH_SIZE    = 64
EPOCHS        = 80
LEARNING_RATE = 1e-3
PATIENCE      = 12       # Early stopping

# --- 出力パス ---
MODELS_DIR  = ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)
MODEL_PATH  = MODELS_DIR / "lstm_bb.pt"
SCALER_PATH = MODELS_DIR / "scaler_bb.pkl"

使用デバイス: cuda


In [ ]:
# ── セル 3: 銘柄リスト読み込み ──────────────────────────────────────────────
TICKERS_PATH = ROOT / "data" / "nikkei225_tickers.txt"

with open(TICKERS_PATH, encoding="utf-8") as f: 
    tickers = [line.strip() for line in f if line.strip()]

print(f"日経225: {len(tickers)} 銘柄")
print("例:", tickers[:5])

UnicodeDecodeError: 'cp932' codec can't decode byte 0x83 in position 101: illegal multibyte sequence

In [ ]:
# ── セル 4: 過去 3 年の株価データを一括ダウンロード ───────────────────────
# ※ 初回は数分かかります。データは変数に保持されます。

print("yfinance でデータダウンロード中...")
raw = yf.download(
    tickers=tickers,
    period="3y",
    interval="1d",
    group_by="ticker",
    auto_adjust=True,
    progress=True,
    threads=True,
)
print(f"ダウンロード完了。データ shape: {raw.shape}")

In [ ]:
# ── セル 5: 特徴量とラベルの抽出関数 ────────────────────────────────────────

def _is_breakout_day(df: pd.DataFrame, t: int, lookback: int, threshold: float) -> bool:
    """
    インデックス t の日がブレイクアウト条件を満たすかチェック。
    bb_scanner._run_scan と同じロジックを使用。
    """
    p0 = df.iloc[t]
    if t < 6:
        return False
    p3 = df.iloc[t - 3]
    p6 = df.iloc[t - 6]

    # 条件1: 現在値が BB 上限付近
    if pd.isna(p0["BB_upper"]) or p0["Close"] < p0["BB_upper"] * threshold:
        return False

    # 条件2: BB 上限が右肩上がり
    if not (p0["BB_upper"] > p3["BB_upper"] > p6["BB_upper"]):
        return False

    # 条件3: BB 中心も上昇中
    if not (p0["BB_middle"] > p3["BB_middle"] > p6["BB_middle"]):
        return False

    # 条件4: バンド幅が拡大中
    bw_now  = float(p0["BB_upper"])  - float(p0["BB_lower"])
    bw_prev = float(p6["BB_upper"]) - float(p6["BB_lower"])
    if bw_now <= bw_prev:
        return False

    # 条件5: lookback 日前は上限に未達（新規ブレイク）
    if t > lookback:
        px = df.iloc[t - lookback]
        if not pd.isna(px["BB_upper"]) and px["Close"] >= px["BB_upper"] * threshold:
            return False

    return True


def extract_samples(
    df: pd.DataFrame,
    seq_len: int = SEQ_LEN,
    horizon: int = LABEL_HORIZON,
    gain: float = LABEL_GAIN,
    bb_period: int = BB_PERIOD,
    threshold: float = BB_THRESHOLD,
    lookback: int = LOOKBACK,
):
    """
    df から全ブレイクアウトイベントを探し、(特徴量配列, ラベル) のリストを返す。

    Returns
    -------
    features : list[np.ndarray]  shape (seq_len, N_FEATURES)
    labels   : list[int]         1 = 成功, 0 = 失敗
    """
    features, labels = [], []
    n = len(df)
    # ブレイクアウト判定できる最小インデックス
    min_t = max(bb_period + lookback, seq_len + 1)

    for t in range(min_t, n - horizon):
        if not _is_breakout_day(df, t, lookback, threshold):
            continue

        # ── 特徴量: t-seq_len 〜 t の seq_len 日分 ──
        seg = df.iloc[t - seq_len : t + 1]   # seq_len+1 行 (pct_change用)
        if len(seg) < seq_len + 1:
            continue

        f1 = seg["Close"].pct_change().values[1:]

        vol_ma_col = next((c for c in df.columns if c.startswith("Vol_M")), None)
        if vol_ma_col:
            vma = seg[vol_ma_col].values[1:]
            with np.errstate(divide="ignore", invalid="ignore"):
                f2 = np.where(vma > 0, seg["Volume"].values[1:] / vma, 1.0)
        else:
            f2 = np.ones(seq_len)

        bw  = (seg["BB_upper"] - seg["BB_lower"]).values[1:]
        with np.errstate(divide="ignore", invalid="ignore"):
            f3 = np.where(
                bw > 0,
                (seg["Close"].values[1:] - seg["BB_lower"].values[1:]) / bw,
                0.5,
            )

        mid = seg["BB_middle"].values[1:]
        with np.errstate(divide="ignore", invalid="ignore"):
            f4 = np.where(mid > 0, bw / mid, 0.0)

        f5 = np.diff(seg["BB_upper"].values) / seg["BB_upper"].values[:-1]

        feat = np.stack([f1, f2, f3, f4, f5], axis=1).astype(np.float32)
        feat = np.nan_to_num(feat, nan=0.0, posinf=3.0, neginf=-3.0)
        feat = np.clip(feat, -5.0, 5.0)

        # ── ラベル: horizon 日以内に +gain% 以上上昇 ──
        future_close = df.iloc[t + 1 : t + 1 + horizon]["Close"]
        if len(future_close) == 0:
            continue
        max_gain = float(future_close.max()) / float(df.iloc[t]["Close"]) - 1
        label = 1 if max_gain >= gain else 0

        features.append(feat)
        labels.append(label)

    return features, labels

print("関数定義 OK")

In [ ]:
# ── セル 6: 全銘柄からサンプル収集 ──────────────────────────────────────────
all_features, all_labels = [], []
min_bars = BB_PERIOD + max(LOOKBACK, 10) + SEQ_LEN + LABEL_HORIZON + 5
single   = len(tickers) == 1

for i, code in enumerate(tickers):
    try:
        df = raw.copy() if single else raw[code].copy()
        if df is None or df.empty:
            continue

        # 列名正規化
        df.columns = [str(c).capitalize() for c in df.columns]
        if df.index.tz is not None:
            df.index = df.index.tz_localize(None)
        df.dropna(subset=["Close"], inplace=True)

        if len(df) < min_bars:
            continue

        df = calc_bollinger_bands(df, period=BB_PERIOD)
        df = calc_volume_ma(df)
        df.dropna(subset=["BB_upper", "BB_middle", "BB_lower"], inplace=True)

        feats, lbls = extract_samples(df)
        all_features.extend(feats)
        all_labels.extend(lbls)

    except Exception as e:
        pass

    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(tickers)} 銘柄処理完了 — サンプル数: {len(all_labels)}")

print(f"\n▶ 総サンプル数 : {len(all_labels)}")
pos = sum(all_labels)
neg = len(all_labels) - pos
print(f"  成功(1)      : {pos} ({pos/len(all_labels)*100:.1f}%)")
print(f"  失敗(0)      : {neg} ({neg/len(all_labels)*100:.1f}%)")

In [ ]:
# ── セル 7: スケーリング & データセット準備 ────────────────────────────────
X = np.array(all_features)  # (N, SEQ_LEN, N_FEATURES)
y = np.array(all_labels, dtype=np.float32)  # (N,)

# StandardScaler を特徴量次元ごとに適用
# flatten → fit → reshape
N, S, F = X.shape
X_flat  = X.reshape(-1, F)          # (N*SEQ_LEN, N_FEATURES)
scaler  = StandardScaler()
X_scaled_flat = scaler.fit_transform(X_flat).astype(np.float32)
X_scaled = X_scaled_flat.reshape(N, S, F)

print(f"X shape: {X_scaled.shape}")

# 学習 80% / 検証 20% に分割（時系列データなので shuffle=False が本来良いが
# 複数銘柄を混ぜているため shuffle=True で問題ない）
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(y_train)}  Val: {len(y_val)}")


class BBDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_loader = DataLoader(BBDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(BBDataset(X_val,   y_val),   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print("DataLoader 準備完了")

In [ ]:
# ── セル 8: モデル・損失・最適化 ────────────────────────────────────────────
model = LSTMClassifier().to(DEVICE)
print(model)

# クラス不均衡対策: 少数クラス（成功）の重みを上げる
pos_count  = float(y_train.sum())
neg_count  = float(len(y_train) - pos_count)
pos_weight = torch.tensor([neg_count / pos_count], device=DEVICE)
print(f"pos_weight: {pos_weight.item():.3f}")

criterion = nn.BCELoss(reduction="mean")  # Sigmoid は model 内部に含まれている
# ↑ BCEWithLogitsLoss を使わず、モデルの Sigmoid 後に BCE をかける

# pos_weight を BCELoss に反映させるため手動で重み付け
def weighted_bce(pred, target):
    weight = torch.where(target == 1, pos_weight, torch.ones_like(pos_weight))
    return nn.functional.binary_cross_entropy(pred, target, weight=weight)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=5, verbose=True
)
print("セットアップ完了")

In [ ]:
# ── セル 9: 学習ループ ───────────────────────────────────────────────────────
best_val_auc  = 0.0
no_improve    = 0
train_losses  = []
val_aucs      = []

for epoch in range(1, EPOCHS + 1):
    # ─ Train ─
    model.train()
    running_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        pred = model(xb)
        loss = weighted_bce(pred, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # 勾配クリップ
        optimizer.step()
        running_loss += loss.item() * len(yb)
    avg_loss = running_loss / len(train_loader.dataset)
    train_losses.append(avg_loss)

    # ─ Validation ─
    model.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(DEVICE)
            pred = model(xb).cpu().numpy()
            preds_all.extend(pred)
            labels_all.extend(yb.numpy())

    val_auc = roc_auc_score(labels_all, preds_all)
    val_aucs.append(val_auc)
    scheduler.step(val_auc)

    # ─ ログ ─
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{EPOCHS}  loss={avg_loss:.4f}  val_AUC={val_auc:.4f}")

    # ─ Early stopping & ベストモデル保存 ─
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        no_improve = 0
        torch.save(model.state_dict(), MODEL_PATH)
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch} (best AUC={best_val_auc:.4f})")
            break

print(f"\n学習完了 — ベスト Val AUC: {best_val_auc:.4f}")

In [ ]:
# ── セル 10: 学習曲線の可視化 ───────────────────────────────────────────────
try:
    import matplotlib.pyplot as plt

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(train_losses)
    ax1.set_title("Training Loss")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Weighted BCE Loss")
    ax1.grid(True)

    ax2.plot(val_aucs)
    ax2.axhline(best_val_auc, color="red", linestyle="--", label=f"Best AUC={best_val_auc:.4f}")
    ax2.set_title("Validation ROC-AUC")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("AUC")
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.savefig(str(MODELS_DIR / "training_curve.png"), dpi=150)
    plt.show()
    print("学習曲線を models/training_curve.png に保存しました")
except ImportError:
    print("matplotlib がインストールされていません。pip install matplotlib でインストールしてください。")

In [ ]:
# ── セル 11: テスト評価（ベストモデルをロードして詳細確認） ─────────────────
# ベストモデルを再ロード
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True))
model.eval()

preds_all, labels_all = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        xb = xb.to(DEVICE)
        pred = model(xb).cpu().numpy()
        preds_all.extend(pred)
        labels_all.extend(yb.numpy())

preds_binary = [1 if p >= 0.5 else 0 for p in preds_all]

print("=== 評価レポート ===")
print(classification_report(labels_all, preds_binary, target_names=["失敗(0)", "成功(1)"]))
print(f"ROC-AUC: {roc_auc_score(labels_all, preds_all):.4f}")
print()

# 確率分布
preds_arr = np.array(preds_all)
print("確率分布（検証データ）:")
print(f"  25パーセンタイル : {np.percentile(preds_arr, 25):.3f}")
print(f"  中央値           : {np.percentile(preds_arr, 50):.3f}")
print(f"  75パーセンタイル : {np.percentile(preds_arr, 75):.3f}")

In [ ]:
# ── セル 12: スケーラー保存 ────────────────────────────────────────────────
with open(SCALER_PATH, "wb") as f:
    pickle.dump(scaler, f)

print(f"モデル保存  : {MODEL_PATH}")
print(f"スケーラー  : {SCALER_PATH}")

# サイズ確認
import os
print(f"モデルサイズ: {os.path.getsize(MODEL_PATH) / 1024:.1f} KB")
print()
print("次のステップ:")
print("  git add models/lstm_bb.pt models/scaler_bb.pkl")
print("  git commit -m 'add: BB LSTM trained model'")
print("  git push")
print()
print("Streamlit Cloud に push すると BB スキャナーで AI 確率が表示されます！")

In [ ]:
# ── セル 13: 動作確認（推論テスト） ──────────────────────────────────────────
# lstm_predictor.py が正しく動くかをノートブック内でテスト
from modules.lstm_predictor import predict_proba, is_model_available

print(f"モデル利用可能: {is_model_available()}")

# ダミーデータで推論テスト（実際のスキャン後に実行されるのと同じ）
# 検証データの最初のサンプルを使って確認
sample_code = tickers[0]
try:
    df_sample = raw.copy() if single else raw[sample_code].copy()
    df_sample.columns = [str(c).capitalize() for c in df_sample.columns]
    if df_sample.index.tz is not None:
        df_sample.index = df_sample.index.tz_localize(None)
    df_sample.dropna(subset=["Close"], inplace=True)
    df_sample = calc_bollinger_bands(df_sample)
    df_sample = calc_volume_ma(df_sample)

    prob = predict_proba(df_sample)
    if prob is not None:
        print(f"{sample_code} の AI 成功確率: {prob:.1f}%")
    else:
        print("確率計算不可（データ不足 or エラー）")
except Exception as e:
    print(f"テストエラー: {e}")

## 完了 🎉

学習が完了しました。以下のコマンドで GitHub に push してください：

```bash
cd a:/プログラミング/kabu_dashboard
git add models/lstm_bb.pt models/scaler_bb.pkl
git commit -m "add: BB LSTM trained model (AUC=XX)"
git push
```

push 後、Streamlit Cloud が自動でデプロイし直し、
BB スキャナーの結果テーブルに **AI確率%** 列が表示されます。

### AUC の読み方
| AUC | 解釈 |
|-----|------|
| 0.50 | ランダム（運任せ）|
| 0.55 | わずかな予測力 |
| 0.60 | 実用レベルの入口 |
| 0.65+ | 良好（株式市場でこれは十分強い）|

### 精度を上げるには
- **特徴量追加**: RSI, MACD, 出来高変化率, 過去リターン
- **データ増加**: Nikkei225 → 東証プライム全銘柄（1800銘柄）
- **期間延長**: `period="3y"` → `start="2018-01-01"`
- **アーキテクチャ変更**: Transformer, TCN, または LightGBM との比較